In [1]:
import pandas as pd
import numpy as np
import h5py
import os


In [10]:
def write_to_hdf5_file(data, output_h5_path, hdr = []):
    '''
    write to h5 file function for icu sleep and sleeplab+airgo data.
    '''
    
    chunk_size = 64

    with h5py.File(output_h5_path, 'a') as f:
        
        # save signals:
        for signal_tmp in data.columns:
            if not signal_tmp in f: # first write of this signal
                
                if signal_tmp in ['Annotation']: 
                    dtype1 = h5py.string_dtype(encoding='utf-8') # h5py needs to be >= 2.10
                    dset_signal = f.create_dataset(signal_tmp, shape=(data.shape[0],), maxshape=(None,),
                                          chunks=(chunk_size,), dtype=dtype1)
                    dset_signal[:] = data[signal_tmp].astype('str')
                    continue # with next signal
                    
                if '_event' in signal_tmp: dtype1 = 'int8'               # for icu-sleep data
                elif signal_tmp in ['Stage','Apnea']: dtype1 = 'int8'    # for sleep lab data
                elif signal_tmp in ['Epoch']: dtype1='int64'              # for sleep lab data
                elif signal_tmp in ['ABD','CHEST']: dtype1='float32'              # for sleep lab data
                else: dtype1 = 'float16'
                    
                dset_signal = f.create_dataset(signal_tmp, shape=(data.shape[0],), maxshape=(None,),
                                          chunks=(chunk_size,), dtype=dtype1)
                dset_signal[:] = data[signal_tmp].astype(dtype1)
                
            else:                
                raise ValueError('Code exists but currently not intended to write on existing h5 file. --> To Check.')
                
                if '_event' in signal_tmp: dtype1 = 'int8'
                else: dtype1 = 'float16'
                    
                f[signal_tmp].resize((f[signal_tmp].shape[0] + data[signal_tmp].shape[0]), axis = 0)
                f[signal_tmp][-data[signal_tmp].shape[0]:] = data[signal_tmp].astype(dtype1)

                
        # save header:
        if hdr:
            for hdr_element in hdr.keys():

                if type(hdr[hdr_element]) == np.int32:
                    dset_signal = f.create_dataset(hdr_element, shape=(1,), maxshape=(1,),
                                              chunks=True, dtype=np.int32)
                    dset_signal[:] = hdr[hdr_element]

                elif type(hdr[hdr_element]) == type(hdr[hdr_element]) == np.ndarray:
                    dset_signal = f.create_dataset(hdr_element, shape=(hdr[hdr_element].shape[0],), maxshape=(hdr[hdr_element].shape[0]+10,),
                                              chunks=True, dtype=np.int32)
                    dset_signal[:] = hdr[hdr_element].astype(np.int32)
                
                elif type(di.id) == str:
                    dset_signal = f.create_dataset(hdr_element, shape=(1,),
                                              chunks=True, dtype='S7')
                    dset_signal[:] = np.array(di.id.encode('utf8'))              
                    
                else:
                    raise ValueError('Unexpected datatype for header.')


In [12]:
def index_to_datetime_sleepdata(data, hdr):
    data.index = pd.date_range(hdr['start_datetime'], periods=data.shape[0], freq=str(1/hdr['fs'])+'S')
    return data

In [13]:
def load_sleep_data(filepath, load_all_signals = 1, idx_to_datetime = 0, signals_to_load = [], load_events = 0, verbose = False):
    '''
    filepath: full filepath of sleep data (icu sleep or sleeplab .h5) file to be loaded
    load_all_signals = 1: boolean if all the data contained in file should be loaded.
    idx_to_datetime = 0: boolean if index should be transformed to datetime. expected: hdr in correct format, see 'write_to_hdf5_file' function.
    signals_to_load = []: non-default only needed if load_all_signals ==0, list of signals that should be loaded
    load_events = 0: if events (bedmaster time correction) should be loaded.
    '''

    # load file object
    ff = h5py.File(filepath, 'r')

    if load_all_signals:
        signals_to_load = list(ff.keys())
    if verbose:
        print(f'signals to load: {signals_to_load}')


    # load the data:
    data = pd.DataFrame([])
    hdr = {}
    for signal_to_load_tmp in signals_to_load:
        if 'event' in signal_to_load_tmp and not load_events: continue
        if signal_to_load_tmp in ['study_id', 'MRN', 'fs']:
            # add to header
            hdr[signal_to_load_tmp] = ff[signal_to_load_tmp][:]
            if hdr[signal_to_load_tmp].shape[0]:
                hdr[signal_to_load_tmp] = hdr[signal_to_load_tmp][0]
        elif signal_to_load_tmp == 'start_datetime':
            t = ff[signal_to_load_tmp][:]
            hdr[signal_to_load_tmp] = pd.to_datetime(f'{t[0]}-{t[1]}-{t[2]} {t[3]}:{t[4]}:{t[5]}.{t[6]}', infer_datetime_format = True)
        elif signal_to_load_tmp == 'day_night_id':
            t = ff[signal_to_load_tmp][:][0]
            hdr[signal_to_load_tmp] = t.decode('utf8')           
        else:
            data[signal_to_load_tmp] = ff[signal_to_load_tmp][:]
        
    ff.close()
    
    if idx_to_datetime:
        data = index_to_datetime_sleepdata(data, hdr)
        
    return data, hdr

# example:
if 0:
    data, hdr = load_sleep_data(output_h5_path)

In [1]:
def sleeplab_to_sleep_research_format(airgo, psg, annotation=None):
    
    data = pd.concat([airgo, psg], join='outer', axis=1, sort=True)

    for column in data.columns:
        if column in ['Stage','Apnea','Epoch']:
            data[column] = data[column].interpolate(method='nearest', limit_area='inside',
                                      limit=15)        
        else:
            data[column] = data[column].interpolate(method='pchip', order=3, limit_area='inside',
                                      limit=15)

    data = data.resample('0.1S').mean()

    if annotation is not None:
        data = data.join(annotation)

    return data

In [6]:
def format_bm_airgo_to_10Hz_icusleep_data(bm_data, airgo_data):
    data_list = [bm_data[signal_tmp] for signal_tmp in bm_data.keys()]
    data_list.append(airgo_data)
    data = pd.concat(data_list, join='outer', axis=1, sort=True)
    for signal_tmp in bm_data.keys():
        data[signal_tmp] = data[signal_tmp].interpolate(method='pchip', order=3, limit_area='inside',
                                  limit=50)
        data[signal_tmp + '_event'] = data[signal_tmp + '_event'].interpolate(method='pchip', order=3, limit_area='inside',
                                  limit=50)
        data.loc[np.isnan(data[signal_tmp + '_event']), signal_tmp + '_event'] = 99 # replace nan with 99 because data gets saved as int in hdf5, nan not supported.

    print('To Check: I believe I need to add  data.resample("0.1S").mean()')
    return data
    # limit considerations: 0.5Hz bedmaster to 10 Hz: for each datapoint, 19 NaNs. 
    # datagaps less than 5 seconds will be interpolated, if larger than 5 seconds, it is NaN. i.e. limit 50.
